In [ ]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time
import tkinter as tk
from threading import Thread, Event
from datetime import datetime

# ------------------------- CONFIGURATION -------------------------
pyautogui.FAILSAFE = False          # Fixed typo
GESTURE_COOLDOWN = 0.5              # seconds between same gesture actions
VOLUME_INCREMENT = 5                # scroll amount or volume steps

# ------------------------- GLOBALS -------------------------
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_draw = mp.solutions.drawing_utils
cap = cv2.VideoCapture(0)

gesture_text = "No Gesture"
mode = "DEFAULT"
last_action_time = 0                 # for cooldown
shutdown_event = Event()             # to stop both threads cleanly

# ------------------------- HELPER FUNCTIONS -------------------------
def get_fingers(lm_list):
    """
    Returns list of 5 booleans (or ints) indicating which fingers are raised.
    Uses y-coordinate (vertical position) for all fingers, including thumb.
    """
    fingers = []
    # Thumb: compare tip y with IP joint y (index 3)
    if lm_list[4][1] < lm_list[3][1]:
        fingers.append(1)
    else:
        fingers.append(0)
    
    # Other fingers: compare tip y with middle joint y
    tips = [8, 12, 16, 20]
    for tip in tips:
        if lm_list[tip][1] < lm_list[tip - 2][1]:
            fingers.append(1)
        else:
            fingers.append(0)
    return fingers

def take_screenshot():
    """Save screenshot with timestamp to avoid overwriting."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"screenshot_{timestamp}.png"
    pyautogui.screenshot(filename)
    print(f"Screenshot saved as {filename}")

# ------------------------- GESTURE DETECTION THREAD -------------------------
def gesture_loop():
    global gesture_text, mode, last_action_time

    while not shutdown_event.is_set():
        success, frame = cap.read()
        if not success:
            continue

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        lm_list = []
        if result.multi_hand_landmarks:
            for handLms in result.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, handLms, mp_hands.HAND_CONNECTIONS)
                h, w, _ = frame.shape
                for id, lm in enumerate(handLms.landmark):
                    lm_list.append((int(w * lm.x), int(h * lm.y)))

        # Gesture recognition with cooldown
        if lm_list:
            fingers = get_fingers(lm_list)
            now = time.time()

            if fingers == [0, 0, 0, 0, 0] and (now - last_action_time) > GESTURE_COOLDOWN:
                gesture_text = "FIST → Play/Pause"
                mode = "MEDIA CONTROL"
                pyautogui.press("space")
                last_action_time = now

            elif fingers == [1, 1, 1, 1, 1] and (now - last_action_time) > GESTURE_COOLDOWN:
                gesture_text = "OPEN PALM → Screenshot"
                mode = "SCREENSHOT"
                take_screenshot()
                last_action_time = now

            elif fingers == [0, 1, 0, 0, 0]:
                gesture_text = "INDEX → Scroll"
                mode = "SCROLL"
                # Scroll with cooldown? Scrolling without cooldown is often desired,
                # but we apply a small delay to avoid hyper‑scroll.
                if (now - last_action_time) > 0.1:
                    pyautogui.scroll(VOLUME_INCREMENT)
                    last_action_time = now

            elif fingers == [0, 1, 1, 0, 0] and (now - last_action_time) > GESTURE_COOLDOWN:
                gesture_text = "TWO FINGERS → Volume Up"
                mode = "VOLUME"
                # Fallback: use key press if volume keys work, else use system volume commands
                pyautogui.press("volumeup")
                last_action_time = now

            # You can add more gestures here (e.g., volume down, browser back, etc.)

        # Overlay text on video feed
        cv2.putText(frame, f"Gesture: {gesture_text}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f"Mode: {mode}", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        cv2.imshow("AI Gesture Control System", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            shutdown_event.set()
            break

    cap.release()
    cv2.destroyAllWindows()

# ------------------------- GUI THREAD -------------------------
def gui_loop():
    global gesture_text, mode
    root = tk.Tk()
    root.title("Gesture Control Dashboard")
    root.geometry("300x200")
    root.protocol("WM_DELETE_WINDOW", lambda: shutdown_event.set())  # Clean exit on close

    label1 = tk.Label(root, text="System Status: ACTIVE", font=("Arial", 12))
    label1.pack(pady=10)

    label2 = tk.Label(root, text="Gesture: None", font=("Arial", 12))
    label2.pack(pady=10)

    label3 = tk.Label(root, text="Mode: DEFAULT", font=("Arial", 12))
    label3.pack(pady=10)

    def update():
        label2.config(text=f"Gesture: {gesture_text}")
        label3.config(text=f"Mode: {mode}")
        if not shutdown_event.is_set():
            root.after(200, update)

    update()
    root.mainloop()
    # When tkinter window is closed, ensure the other thread knows
    shutdown_event.set()

# ------------------------- MAIN -------------------------
if __name__ == "__main__":
    t1 = Thread(target=gesture_loop)
    t2 = Thread(target=gui_loop)

    t1.start()
    t2.start()

    # Wait for both threads to finish (shutdown_event triggered)
    t1.join()
    t2.join()

    print("Gesture control system stopped.")

Screenshot saved as screenshot_20260522_105407.png
Screenshot saved as screenshot_20260522_105409.png
Screenshot saved as screenshot_20260522_105410.png
Screenshot saved as screenshot_20260522_105410.png
Screenshot saved as screenshot_20260522_105411.png
Screenshot saved as screenshot_20260522_105411.png
Screenshot saved as screenshot_20260522_105413.png
Screenshot saved as screenshot_20260522_105414.png
Screenshot saved as screenshot_20260522_105415.png
Screenshot saved as screenshot_20260522_105415.png
Screenshot saved as screenshot_20260522_105416.png
Screenshot saved as screenshot_20260522_105416.png
Screenshot saved as screenshot_20260522_105417.png
Screenshot saved as screenshot_20260522_105424.png
Screenshot saved as screenshot_20260522_105425.png
Screenshot saved as screenshot_20260522_105428.png
Screenshot saved as screenshot_20260522_105429.png
Screenshot saved as screenshot_20260522_105430.png
Screenshot saved as screenshot_20260522_105430.png
Screenshot saved as screenshot_

In [ ]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time
import tkinter as tk
from tkinter import ttk
from threading import Thread, Event
from datetime import datetime
import platform

# ─────────────────────────────────────────────────────────────
#  PLATFORM-AWARE VOLUME CONTROL
# ─────────────────────────────────────────────────────────────
SYSTEM = platform.system()   # "Windows" | "Darwin" | "Linux"

def volume_up():
    if SYSTEM == "Windows":
        pyautogui.press("volumeup")
    elif SYSTEM == "Darwin":
        pyautogui.hotkey("shift", "option", "volumeup")   # finer step on Mac
    else:
        import subprocess
        subprocess.call(["amixer", "-D", "pulse", "sset", "Master", "5%+"])

def volume_down():
    if SYSTEM == "Windows":
        pyautogui.press("volumedown")
    elif SYSTEM == "Darwin":
        pyautogui.hotkey("shift", "option", "volumedown")
    else:
        import subprocess
        subprocess.call(["amixer", "-D", "pulse", "sset", "Master", "5%-"])

def volume_mute():
    if SYSTEM == "Windows":
        pyautogui.press("volumemute")
    elif SYSTEM == "Darwin":
        pyautogui.hotkey("option", "F10")
    else:
        import subprocess
        subprocess.call(["amixer", "-D", "pulse", "sset", "Master", "toggle"])


# ─────────────────────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────────────────────
pyautogui.FAILSAFE = False

GESTURE_COOLDOWN   = 0.6    # seconds between triggered actions
SCROLL_COOLDOWN    = 0.12   # faster repeat for scroll
VOLUME_COOLDOWN    = 0.4    # volume repeat rate

# ─────────────────────────────────────────────────────────────
#  MEDIAPIPE SETUP
# ─────────────────────────────────────────────────────────────
mp_hands = mp.solutions.hands
hands    = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.6
)
mp_draw = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

cap = cv2.VideoCapture(0)

# ─────────────────────────────────────────────────────────────
#  SHARED STATE
# ─────────────────────────────────────────────────────────────
gesture_text   = "No Gesture"
mode           = "IDLE"
last_action_time = 0
shutdown_event = Event()
finger_count   = 0          # live finger count shown in GUI


# ═════════════════════════════════════════════════════════════
#  HELPER: FINGER DETECTION
# ═════════════════════════════════════════════════════════════
def get_finger_states(lm_list):
    """
    Returns list of 5 bools: [thumb, index, middle, ring, pinky]
    True = finger is raised.
    """
    fingers = []

    # Thumb — compare tip (4) vs IP joint (3) on x-axis
    # Works for right hand facing camera (mirrored)
    if lm_list[4][0] > lm_list[3][0]:
        fingers.append(1)
    else:
        fingers.append(0)

    # Four fingers — compare tip y vs 2nd-knuckle y
    for tip_id in [8, 12, 16, 20]:
        if lm_list[tip_id][1] < lm_list[tip_id - 2][1]:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers


def count_fingers(fingers):
    return sum(fingers)


def take_screenshot():
    ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"screenshot_{ts}.png"
    pyautogui.screenshot(filename)
    print(f"[SCREENSHOT] Saved: {filename}")
    return filename


# ═════════════════════════════════════════════════════════════
#  GESTURE → ACTION MAPPING
# ═════════════════════════════════════════════════════════════
#
#  Fingers raised  |  Action
#  ─────────────────────────────────────────────────────────
#  0  (fist)       |  Play / Pause  (spacebar)
#  1  (index)      |  Volume UP
#  2  (index+mid)  |  Volume DOWN
#  3               |  Scroll UP
#  4               |  Scroll DOWN
#  5  (open palm)  |  Screenshot
#  Thumb only      |  Mute toggle
#
# ─────────────────────────────────────────────────────────────

def process_gesture(fingers, now):
    global gesture_text, mode, last_action_time, finger_count

    count = count_fingers(fingers)
    finger_count = count

    # ── Thumb only (all others down) → Mute ────────────────
    if fingers == [1, 0, 0, 0, 0]:
        if (now - last_action_time) > GESTURE_COOLDOWN:
            gesture_text = "👍 THUMB → Mute Toggle"
            mode = "VOLUME"
            volume_mute()
            last_action_time = now
        return

    # ── 0 fingers (fist) → Play/Pause ──────────────────────
    if count == 0:
        if (now - last_action_time) > GESTURE_COOLDOWN:
            gesture_text = "✊ FIST → Play / Pause"
            mode = "MEDIA"
            pyautogui.press("space")
            last_action_time = now
        return

    # ── 1 finger (index only) → Volume UP ──────────────────
    if fingers[1] == 1 and count == 1:
        if (now - last_action_time) > VOLUME_COOLDOWN:
            gesture_text = "☝️ 1 FINGER → Volume UP"
            mode = "VOLUME ▲"
            volume_up()
            last_action_time = now
        return

    # ── 2 fingers (index + middle) → Volume DOWN ───────────
    if fingers[1] == 1 and fingers[2] == 1 and count == 2:
        if (now - last_action_time) > VOLUME_COOLDOWN:
            gesture_text = "✌️ 2 FINGERS → Volume DOWN"
            mode = "VOLUME ▼"
            volume_down()
            last_action_time = now
        return

    # ── 3 fingers → Scroll UP ──────────────────────────────
    if count == 3:
        if (now - last_action_time) > SCROLL_COOLDOWN:
            gesture_text = "3 FINGERS → Scroll UP"
            mode = "SCROLL ▲"
            pyautogui.scroll(5)
            last_action_time = now
        return

    # ── 4 fingers → Scroll DOWN ────────────────────────────
    if count == 4 and fingers[0] == 0:
        if (now - last_action_time) > SCROLL_COOLDOWN:
            gesture_text = "4 FINGERS → Scroll DOWN"
            mode = "SCROLL ▼"
            pyautogui.scroll(-5)
            last_action_time = now
        return

    # ── 5 fingers (open palm) → Screenshot ─────────────────
    if count == 5:
        if (now - last_action_time) > GESTURE_COOLDOWN:
            gesture_text = "🖐 OPEN PALM → Screenshot"
            mode = "SCREENSHOT"
            take_screenshot()
            last_action_time = now
        return

    # Default
    gesture_text = f"Fingers: {count}"
    mode = "IDLE"


# ═════════════════════════════════════════════════════════════
#  GESTURE DETECTION THREAD (OpenCV)
# ═════════════════════════════════════════════════════════════
def gesture_loop():
    global gesture_text, mode

    # Color scheme for landmarks
    hand_spec       = mp_draw.DrawingSpec(color=(0, 255, 180), thickness=2, circle_radius=4)
    connection_spec = mp_draw.DrawingSpec(color=(255, 255, 255), thickness=2)

    while not shutdown_event.is_set():
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.flip(frame, 1)
        h, w  = frame.shape[:2]
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        lm_list = []
        if result.multi_hand_landmarks:
            for hand_lms in result.multi_hand_landmarks:
                mp_draw.draw_landmarks(
                    frame, hand_lms,
                    mp_hands.HAND_CONNECTIONS,
                    hand_spec, connection_spec
                )
                for lm in hand_lms.landmark:
                    lm_list.append((int(w * lm.x), int(h * lm.y)))

        if lm_list:
            fingers = get_finger_states(lm_list)
            process_gesture(fingers, time.time())
        else:
            gesture_text = "No Hand Detected"
            mode = "IDLE"

        # ── HUD overlay ─────────────────────────────────────
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 100), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.45, frame, 0.55, 0, frame)

        cv2.putText(frame, gesture_text,
                    (15, 38), cv2.FONT_HERSHEY_DUPLEX, 0.75, (0, 255, 160), 2)
        cv2.putText(frame, f"Mode: {mode}",
                    (15, 72), cv2.FONT_HERSHEY_DUPLEX, 0.65, (100, 200, 255), 2)

        # Finger count badge
        badge_text = str(finger_count)
        cv2.circle(frame, (w - 45, 45), 35, (255, 180, 0), -1)
        cv2.putText(frame, badge_text,
                    (w - 58, 58), cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 0, 0), 3)

        # Hint bar at bottom
        hint = "q=quit | 1=Vol+ | 2=Vol- | 0=Play/Pause | 5=Screenshot"
        cv2.rectangle(frame, (0, h - 30), (w, h), (0, 0, 0), -1)
        cv2.putText(frame, hint,
                    (8, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1)

        cv2.imshow("AI Gesture Control System", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            shutdown_event.set()
            break

    cap.release()
    cv2.destroyAllWindows()


# ═════════════════════════════════════════════════════════════
#  DASHBOARD GUI THREAD (Tkinter)
# ═════════════════════════════════════════════════════════════
def gui_loop():
    root = tk.Tk()
    root.title("Gesture Control Dashboard")
    root.geometry("420x480")
    root.configure(bg="#0d0d0d")
    root.resizable(False, False)
    root.protocol("WM_DELETE_WINDOW", lambda: shutdown_event.set())

    # ── Fonts ────────────────────────────────────────────────
    font_title  = ("Courier New", 14, "bold")
    font_label  = ("Courier New", 11)
    font_value  = ("Courier New", 12, "bold")
    font_small  = ("Courier New", 9)

    # ── Header ───────────────────────────────────────────────
    tk.Label(root, text="◈ GESTURE CONTROL SYSTEM ◈",
             bg="#0d0d0d", fg="#00ff9f", font=font_title).pack(pady=(18, 4))
    tk.Label(root, text=f"Platform: {SYSTEM}",
             bg="#0d0d0d", fg="#555555", font=font_small).pack()

    ttk.Separator(root, orient="horizontal").pack(fill="x", padx=20, pady=10)

    # ── Status ───────────────────────────────────────────────
    status_frame = tk.Frame(root, bg="#0d0d0d")
    status_frame.pack(pady=4)
    tk.Label(status_frame, text="STATUS", bg="#0d0d0d",
             fg="#555", font=font_small).grid(row=0, column=0, padx=20)
    status_val = tk.Label(status_frame, text="● ACTIVE",
                          bg="#0d0d0d", fg="#00ff9f", font=font_value)
    status_val.grid(row=1, column=0, padx=20)

    # ── Live values ──────────────────────────────────────────
    live_frame = tk.Frame(root, bg="#111111", bd=0, relief="flat")
    live_frame.pack(fill="x", padx=20, pady=10)
    live_frame.configure(highlightbackground="#222", highlightthickness=1)

    def make_live_row(parent, label_text, row):
        tk.Label(parent, text=label_text, bg="#111111",
                 fg="#888", font=font_small).grid(
                     row=row*2, column=0, sticky="w", padx=14, pady=(8,0))
        val = tk.Label(parent, text="—", bg="#111111",
                       fg="#ffffff", font=font_value)
        val.grid(row=row*2+1, column=0, sticky="w", padx=14, pady=(0,8))
        return val

    gesture_val = make_live_row(live_frame, "GESTURE DETECTED", 0)
    mode_val    = make_live_row(live_frame, "CURRENT MODE", 1)
    fingers_val = make_live_row(live_frame, "FINGERS UP", 2)

    ttk.Separator(root, orient="horizontal").pack(fill="x", padx=20, pady=8)

    # ── Gesture reference ────────────────────────────────────
    tk.Label(root, text="GESTURE REFERENCE",
             bg="#0d0d0d", fg="#555", font=font_small).pack()

    ref_frame = tk.Frame(root, bg="#0a0a0a")
    ref_frame.pack(fill="x", padx=20, pady=6)

    gestures = [
        ("✊  0 fingers", "Play / Pause",   "#ff6b6b"),
        ("☝️  1 finger",  "Volume UP",       "#00ff9f"),
        ("✌️  2 fingers", "Volume DOWN",     "#00cfff"),
        ("   3 fingers", "Scroll UP",       "#ffd700"),
        ("   4 fingers", "Scroll DOWN",     "#ffa500"),
        ("🖐  5 fingers", "Screenshot",      "#cc99ff"),
        ("👍  Thumb only","Mute Toggle",     "#ff9fdf"),
    ]

    for i, (sign, action, color) in enumerate(gestures):
        bg = "#111" if i % 2 == 0 else "#0d0d0d"
        row = tk.Frame(ref_frame, bg=bg)
        row.pack(fill="x")
        tk.Label(row, text=sign,   bg=bg, fg=color,
                 font=font_small, width=16, anchor="w").pack(side="left", padx=8, pady=3)
        tk.Label(row, text=action, bg=bg, fg="#cccccc",
                 font=font_small).pack(side="left")

    ttk.Separator(root, orient="horizontal").pack(fill="x", padx=20, pady=8)
    tk.Label(root, text='Press  "q"  in camera window to quit',
             bg="#0d0d0d", fg="#444", font=font_small).pack(pady=(0,12))

    # ── Update loop ──────────────────────────────────────────
    def update():
        gesture_val.config(text=gesture_text)
        mode_val.config(text=mode)
        fingers_val.config(text=str(finger_count))

        # Color mode label dynamically
        color_map = {
            "VOLUME ▲" : "#00ff9f",
            "VOLUME ▼" : "#00cfff",
            "SCROLL ▲" : "#ffd700",
            "SCROLL ▼" : "#ffa500",
            "MEDIA"    : "#ff6b6b",
            "SCREENSHOT": "#cc99ff",
        }
        mode_val.config(fg=color_map.get(mode, "#ffffff"))

        if not shutdown_event.is_set():
            root.after(150, update)

    update()
    root.mainloop()
    shutdown_event.set()




if __name__ == "__main__":
    print(f"[INFO] Platform : {SYSTEM}")
    print("[INFO] Starting Gesture Control System...")
    print("[INFO] Press 'q' in the camera window to quit.\n")

    t1 = Thread(target=gesture_loop, daemon=True)
    t2 = Thread(target=gui_loop,     daemon=True)

    t1.start()
    t2.start()

    # Keep main thread alive until shutdown
    while not shutdown_event.is_set():
        time.sleep(0.2)

    t1.join(timeout=2)
    t2.join(timeout=2)
    print("\n[DONE] Gesture control system stopped.")

[INFO] Platform : Windows
[INFO] Starting Gesture Control System...
[INFO] Press 'q' in the camera window to quit.

[SCREENSHOT] Saved: screenshot_20260522_105706.png
[SCREENSHOT] Saved: screenshot_20260522_105713.png
[SCREENSHOT] Saved: screenshot_20260522_105714.png
[SCREENSHOT] Saved: screenshot_20260522_105714.png
[SCREENSHOT] Saved: screenshot_20260522_105715.png
[SCREENSHOT] Saved: screenshot_20260522_105716.png
[SCREENSHOT] Saved: screenshot_20260522_105716.png
